In [0]:


from pyspark.sql.types import (
    StructType, StructField,
    StringType, TimestampType, LongType
)

log_schema = StructType([
    StructField("start_date", TimestampType(), True),
    StructField("end_date", TimestampType(), True),
    StructField("job_link", StringType(), True),
    StructField("status", StringType(), True),
    StructField("run_id", LongType(), True),
    StructField("job_id", LongType(), True)
])

log_df = spark.createDataFrame([], log_schema)

log_df.write.format("delta").mode("overwrite").saveAsTable("log_error_details")


In [0]:
dbutils.widgets.text("run_id","")
dbutils.widgets.text("job_id","")
dbutils.widgets.text("operation","log_start")


In [0]:

run_id = int(dbutils.widgets.get("run_id"))
job_id = int(dbutils.widgets.get("job_id"))
operation = dbutils.widgets.get("operation")

print("Run ID:", run_id)
print("Job ID:", job_id)
print("Operation:", operation)


In [0]:
workspace_url = "https://dbc-5c07a240-0780.cloud.databricks.com"

job_link = (
    f"{workspace_url}/jobs/{job_id}/runs/{run_id}"
    f"?o=7474644504186528"
)

print("Job Link:", job_link)

In [0]:

from pyspark.sql.functions import current_timestamp, lit


if operation == "log_start":

    df1 = spark.createDataFrame(
        [(run_id, job_id, "Started")],
        ["run_id", "job_id", "status"]
    )

    df2 = df1.withColumn("start_date", current_timestamp())

    df3 = df2.withColumn(
        "end_date", lit(None).cast("timestamp")
    )

    df4 = df3.withColumn(
        "job_link",
        lit(job_link)
    )

    df5 = df4.select(
        "start_date",
        "end_date",
        "job_link",
        "status",
        "run_id",
        "job_id"
    )

    df5.write.format("delta").mode("append").saveAsTable("log_error_details")

elif operation == "log_pass":

    spark.sql(f"""
        INSERT INTO log_error_details
        VALUES (
            current_timestamp(),
            current_timestamp(),
            '{job_link}',
            'PASS',
            {run_id},
            {job_id}
        )
    """)
   
elif operation == "log_fail":

    spark.sql(f"""
        INSERT INTO log_error_details
        VALUES (
            current_timestamp(),
            current_timestamp(),
            '{job_link}',
            'FAIL',
            {run_id},
            {job_id}
        )
    """)


In [0]:

    display(spark.sql("select * from log_error_details ORDER BY start_date DESC"))